# Final RAG System — Kaggle Experiment Notebook

## Separate from the baseline

This notebook is the **final-system workflow**. It does not overwrite the baseline notebook or the baseline metrics. The final report will compare this version with the preserved 20-question baseline.

The workflow is: build a reproducible corpus and index, run the final RAG system, save exact retrieved evidence, evaluate with official Ragas v0.4.3, and export all artifacts. Enable Internet and a T4 GPU before running.


## 0. Experiment configuration

Use the development set while improving the system. The 120-question final test set can only be used once every question has a validated reference answer and annotated chunk IDs. Do not tune retrieval, prompts, or models on the final test set.


In [ ]:
from pathlib import Path
import json
import os
import random
import shutil
import subprocess
import sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

REPO_URL = "https://github.com/DAHANIElkhalil25/rag-pipeline.git"
WORK_DIR = Path("/kaggle/working/rag-pipeline-final")
DATA_DIR = Path("/kaggle/working/rag_final_data")

# Start with development records. Switch to test_dataset_v1.jsonl only after
# completing and validating the 120-question annotation template.
EVALUATION_SPLIT = "dev"  # "dev" or "test"
RUN_OFFICIAL_RAGAS = False   # Set True only after configuring a Kaggle Secret.
JUDGE_PROVIDER = "openai"  # "openai" or "mistral"
JUDGE_MODEL = "gpt-4o-mini"  # Example only; use the model your secret supports.
JUDGE_SECRET_NAME = "OPENAI_API_KEY"  # Kaggle Secret name, never write the key in this notebook.
RETRIEVAL_PROFILE = "multilingual_hybrid_rerank"
# Available profiles: "baseline", "multilingual_hybrid", "multilingual_hybrid_rerank".
# Compare profiles only on development questions, then freeze this choice before final testing.
RUN_RETRIEVAL_PROFILE_COMPARISON = False
PREPARE_ANNOTATION_CANDIDATES = False
LAUNCH_UI = False

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORK_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=WORK_DIR, check=True)

os.environ["RAG_DATA_DIR"] = str(DATA_DIR)
os.environ["RAG_RETRIEVAL_PROFILE"] = RETRIEVAL_PROFILE
sys.path.insert(0, str(WORK_DIR))
os.chdir(WORK_DIR)

for module_name in ["config", "etape5_generation", "evaluation.ragas_runner"]:
    sys.modules.pop(module_name, None)

from config import BENCHMARK_DIR, CLEAN_DIR, EVALUATION_DIR, METADATA_DIR, VECTORSTORE_DIR, init_directories
init_directories()

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print(f"Commit: {commit}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Final-system data directory: {DATA_DIR}")


## 1. Build the final reproducible corpus and index

The final pipeline now removes duplicate cleaned documents physically, writes stable chunk/document IDs, and saves an index manifest. These artifacts are essential for deterministic reference-context evaluation.


In [ ]:
from etape1_collecte import main as run_collection
from etape2_nettoyage import main as run_cleaning
from etape3_benchmarking import main as run_benchmarking
from etape4_indexation import main as run_indexing

if not (METADATA_DIR / "corpus_index.csv").exists():
    run_collection()
if not (CLEAN_DIR / "corpus_cleaned_index.csv").exists():
    run_cleaning()
if not (BENCHMARK_DIR / "benchmark_report.json").exists():
    run_benchmarking()
index_manifest_path = VECTORSTORE_DIR / "index_manifest.json"
must_rebuild_index = True
if index_manifest_path.exists():
    previous_manifest = json.loads(index_manifest_path.read_text(encoding="utf-8"))
    previous_profile = previous_manifest.get("index_config", {}).get("retrieval_profile")
    must_rebuild_index = previous_profile != RETRIEVAL_PROFILE
    print(f"Existing index profile: {previous_profile}; requested profile: {RETRIEVAL_PROFILE}")
if must_rebuild_index:
    run_indexing()

index_manifest = json.loads(index_manifest_path.read_text(encoding="utf-8"))
print(json.dumps(index_manifest, ensure_ascii=False, indent=2))


## 2. Prepare evaluation data

`dev_dataset_v1.jsonl` contains the 20 historic questions and is used only for development. The repository also creates a 120-question source-grounded draft bank. Each item still needs human review and final-index chunk IDs before it can become the frozen final JSONL test set.


In [ ]:
from evaluation.bootstrap_datasets import create_development_set, create_final_annotation_template
from evaluation.build_question_bank import build_records, PYTHON_ITEMS, SKLEARN_ITEMS, LANGCHAIN_ITEMS
from evaluation.dataset_schema import write_jsonl

create_development_set()
create_final_annotation_template()

datasets_dir = WORK_DIR / "evaluation" / "datasets"
dev_dataset = datasets_dir / "dev_dataset_v1.jsonl"
test_template = datasets_dir / "test_dataset_v1_annotation_template.csv"
test_draft = datasets_dir / "test_dataset_v1_source_grounded_draft.jsonl"
test_dataset = datasets_dir / "test_dataset_v1.jsonl"

if not test_draft.exists():
    write_jsonl(test_draft, build_records("python", PYTHON_ITEMS) + build_records("scikit_learn", SKLEARN_ITEMS) + build_records("langchain", LANGCHAIN_ITEMS))
print(f"Source-grounded 120-question draft: {test_draft}")

dataset_path = dev_dataset if EVALUATION_SPLIT == "dev" else test_dataset
if EVALUATION_SPLIT == "test" and not test_dataset.exists():
    raise FileNotFoundError(
        f"Complete {test_template.name}, then run evaluation/convert_annotations.py before selecting the final test split."
    )
print(f"Evaluation dataset: {dataset_path}")


## 3. Load the RAG system once

The generator remains local Mistral 7B in 4-bit mode. The pipeline records the exact ranked text windows supplied to Mistral, so Ragas evaluates the evidence that generation actually saw.


In [ ]:
from etape5_generation import load_pipeline

pipeline = load_pipeline()
print(f"Indexed chunks: {len(pipeline.chunks)}")
print(f"Retrieval configuration: {pipeline.search_config}")


## 3a. Optional final-test annotation candidates

This cell retrieves ten candidate chunks for every source-grounded draft question. It does not validate any record automatically. A reviewer must select only chunks that directly support the reference answer before the final CSV is converted to JSONL.


In [ ]:
if PREPARE_ANNOTATION_CANDIDATES:
    from evaluation.create_annotation_candidates import create_candidates
    candidate_path = DATA_DIR / "evaluation" / "annotation_candidates" / "test_dataset_v1_candidates.jsonl"
    create_candidates(pipeline, test_draft, candidate_path, k=10)
    print(f"Reviewable candidate file: {candidate_path}")
else:
    print("Annotation-candidate generation disabled. Enable it only when you are ready to review final chunk IDs.")


## 3b. Optional controlled retrieval-profile comparison

This development-only experiment records top retrieved chunk IDs for the 20 historical questions under each profile. It does not touch the 120-question final test set and it does not run the LLM judge. Use it to inspect retrieval behaviour, then freeze one profile before final evaluation.


In [ ]:
if RUN_RETRIEVAL_PROFILE_COMPARISON:
    from etape4_indexation import main as rebuild_index
    from etape5_generation import load_pipeline
    from evaluation.dataset_schema import read_jsonl

    profiles_to_compare = ["baseline", "multilingual_hybrid", "multilingual_hybrid_rerank"]
    development_questions = read_jsonl(dev_dataset)
    comparison_rows = []
    for profile_name in profiles_to_compare:
        os.environ["RAG_RETRIEVAL_PROFILE"] = profile_name
        rebuild_index()
        candidate_pipeline = load_pipeline()
        for item in development_questions:
            contexts = candidate_pipeline.retrieve(item["user_input"], k=5)
            comparison_rows.append({
                "profile": profile_name,
                "question_id": item["question_id"],
                "retrieved_context_ids": [context.get("chunk_id") for context in contexts],
                "sources": [context.get("doc_source") for context in contexts],
                "scores": [context.get("retrieval_score") for context in contexts],
            })
    comparison_dir = DATA_DIR / "evaluation" / "retrieval_profile_comparison"
    comparison_dir.mkdir(parents=True, exist_ok=True)
    comparison_path = comparison_dir / "development_candidates.json"
    comparison_path.write_text(json.dumps(comparison_rows, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Saved {len(comparison_rows)} development retrieval rows to {comparison_path}")
    os.environ["RAG_RETRIEVAL_PROFILE"] = RETRIEVAL_PROFILE
else:
    print("Profile comparison disabled. Set RUN_RETRIEVAL_PROFILE_COMPARISON=True only for development experiments.")


## 4. Qualitative evidence check

This small check is not a metric. It lets you confirm that answers cite sources and that stable chunk IDs, source URLs, ranks, and prompt context are recorded correctly.


In [ ]:
demo_questions = [
    "Qu'est-ce qu'un décorateur en Python ?",
    "Quelle est la différence entre fit() et fit_transform() ?",
    "Quel est le rôle d'un Agent dans LangChain ?",
]

demo_rows = []
for question in demo_questions:
    result = pipeline.answer(question)
    demo_rows.append({
        "question": question,
        "answer": result.get("answer", ""),
        "context_ids": [item.get("chunk_id") for item in result.get("retrieved_chunks", [])],
        "sources": result.get("sources", []),
    })
    print(f"\nQuestion: {question}\n{result.get('answer', '')[:900]}\n")

run_preview_dir = DATA_DIR / "evaluation" / "qualitative"
run_preview_dir.mkdir(parents=True, exist_ok=True)
(run_preview_dir / "demo_results.json").write_text(json.dumps(demo_rows, ensure_ascii=False, indent=2), encoding="utf-8")


## 5. Official Ragas evaluation

The evaluation judge is deliberately separate from the local generator. Add `OPENAI_API_KEY` or `MISTRAL_API_KEY` as a Kaggle Secret and set `RUN_OFFICIAL_RAGAS = True`. The runner resumes from saved samples after an interruption and writes per-question results, metric reasons/errors, a CSV, and a JSON summary.


In [ ]:
from evaluation.ragas_runner import run_final_evaluation

run_result = None
if RUN_OFFICIAL_RAGAS:
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret(JUDGE_SECRET_NAME)
    run_id = f"{EVALUATION_SPLIT}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
    run_result = await run_final_evaluation(
        pipeline=pipeline,
        dataset_path=dataset_path,
        run_id=run_id,
        provider=JUDGE_PROVIDER,
        judge_model=JUDGE_MODEL,
        api_key=api_key,
    )
    print(json.dumps(run_result["summary"], ensure_ascii=False, indent=2))
else:
    print("Official Ragas is disabled. Configure a Kaggle Secret and set RUN_OFFICIAL_RAGAS=True when ready.")


## 6. Baseline-versus-final comparison

This chart is created only when a final Ragas run is complete. It preserves the baseline values rather than overwriting them. Compare like with like: the old custom evaluator is useful as a diagnostic baseline, while the final Ragas metrics are the academically defensible primary final results.


In [ ]:
import matplotlib.pyplot as plt

baseline_path = WORK_DIR / "evaluation" / "baseline" / "baseline_metrics_v1.json"
baseline = json.loads(baseline_path.read_text(encoding="utf-8"))

if run_result is not None:
    baseline_metrics = baseline["answer_oriented_evaluation"]
    final_metrics = run_result["summary"]["metric_summary"]
    common = [name for name in baseline_metrics if name in final_metrics]
    baseline_values = [baseline_metrics[name] for name in common]
    final_values = [final_metrics[name]["mean"] for name in common]

    x = np.arange(len(common))
    width = 0.36
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - width / 2, baseline_values, width, label="Baseline custom evaluator", color="#8B5E3C")
    ax.bar(x + width / 2, final_values, width, label="Final official Ragas", color="#2358C5")
    ax.set_xticks(x, common, rotation=15, ha="right")
    ax.set_ylim(0, 1)
    ax.set_ylabel("Mean score")
    ax.set_title("Baseline versus final-system evaluation")
    ax.legend()
    fig.tight_layout()
    comparison_path = Path(run_result["run_dir"]) / "baseline_vs_final.png"
    fig.savefig(comparison_path, dpi=180)
    plt.show()
    print(f"Comparison chart: {comparison_path}")
else:
    print("Run official Ragas first to generate the comparison chart.")


## 7. Gradio interface

The interface is optional and reuses the same loaded pipeline. It shows the answer and ranked sources, but it does not change the evaluation artifacts.


In [ ]:
if LAUNCH_UI:
    from config import UI_CONFIG
    from ui import launch_ui
    demo = launch_ui(
        pipeline,
        share=UI_CONFIG["share"],
        server_name=UI_CONFIG["server_name"],
        server_port=UI_CONFIG["server_port"],
    )


## 8. Export reproducible artifacts

Download `rag_final_results_bundle.zip` from Kaggle after the run. It contains corpus/index manifests, final run artifacts, and the baseline record needed for the internship report.


In [ ]:
manifest = {
    "repository": REPO_URL,
    "commit": commit,
    "seed": SEED,
    "evaluation_split": EVALUATION_SPLIT,
    "dataset_path": str(dataset_path),
    "ragas_enabled": RUN_OFFICIAL_RAGAS,
    "run_dir": run_result["run_dir"] if run_result else None,
    "files": sorted(str(path.relative_to(DATA_DIR)) for path in DATA_DIR.rglob("*") if path.is_file()),
}
(DATA_DIR / "final_run_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

archive_dir = Path("/kaggle/working/rag_final_results_bundle")
if archive_dir.exists():
    shutil.rmtree(archive_dir)
shutil.copytree(DATA_DIR, archive_dir)
shutil.make_archive("/kaggle/working/rag_final_results_bundle", "zip", archive_dir)
print("Archive created: /kaggle/working/rag_final_results_bundle.zip")
